In [1]:
import optuna
import axelrod
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
from collections import Counter
import pandas as pd
from math import exp
import numpy as np


c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100
    

In [3]:
def build_fuzzy_player(params):
    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')
    

    _cooperation.automf(names=["low", "medium", "high"])
    _adaptivity.automf(names=["no", "yes"])
    _forgiveness.automf(names=["low", "medium", "high"])
    _forgiveness['low'] = fuzz.gaussmf(_forgiveness.universe, 0, 25)
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe, [25, 50, 75])
    _stochastic.automf(names=["none", "sometimes", "always"])

    # Resulting strategy MFs — also being optimized
    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [
        0,
        params['D_b'],
        params['D_c']
    ])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [
        params['C_a'],
        params['C_b'],
        100
    ])

    # Rebuild rules using the fresh variables above
    rule1 = ctrl.Rule(
        _cooperation['high'] & _adaptivity['no'] & (_forgiveness['medium'] | _forgiveness['high']),
        _resulting_strategy['D']
    )
    rule2 = ctrl.Rule(
        _forgiveness['low'] & _cooperation['high'],
        _resulting_strategy['C']
    )
    rule3 = ctrl.Rule(
        _stochastic['always'] | _adaptivity['no'],
        _resulting_strategy['D']
    )
    rule4 = ctrl.Rule(
        _cooperation['low'] | (_cooperation['medium'] & _forgiveness['low']),
        _resulting_strategy['D']
    )
    rule5 = ctrl.Rule(
        _cooperation['medium'] & _forgiveness['medium'] & _adaptivity['yes'],
        _resulting_strategy['C']
    )

    strategy_ctrl  = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
    _chosen_strategy = ctrl.ControlSystemSimulation(strategy_ctrl)

    # Build the player class dynamically, capturing everything in closure
    class OptimizedFuzzy(Player):

        # Override class-level FIS components with the fresh ones
        cooperation = _cooperation
        adaptivity = _adaptivity
        stochastic = _stochastic
        forgiveness = _forgiveness
        resulting_strategy = _resulting_strategy
        chosen_strategy = _chosen_strategy

        d_thresh = params['d_threshold']
        c_thresh = params['c_threshold']

        # Reset state so trials don't bleed into each other
        first_time = True
        h = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: axelrod.Player) -> Action:

            if len(self.history) == 0 or D not in opponent.history:
                return C

            coop  = FuzzyMethods.calc_cooperation(self, opponent)
            adap  = FuzzyMethods.calc_adaptivity(self, opponent)
            forg  = FuzzyMethods.calc_forgiveness(self, opponent)
            stoch = FuzzyMethods.calc_stochastic(self, opponent)

            self.chosen_strategy.input['cooperation'] = coop
            self.chosen_strategy.input['adaptivity']  = adap
            self.chosen_strategy.input['forgiveness'] = forg
            self.chosen_strategy.input['stochastic']  = stoch

            try:
                self.chosen_strategy.compute()
                output_val = self.chosen_strategy.output['resulting_strategy']
            except KeyError:
                # No rules fired — default to cooperate
                return C
            except Exception:
                return C

            d_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['D'].mf,
                output_val
            )
            c_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['C'].mf,
                output_val
            )

            if d_membership >= self.d_thresh and c_membership < self.c_thresh:
                return D

            return C

    return OptimizedFuzzy()

In [4]:

def objective(trial):

    D_a = 0
    D_b = trial.suggest_int('D_b', 1, 49)
    D_c = trial.suggest_int('D_c', D_b, 60)

    C_a = trial.suggest_int('C_a', 25, 74)
    C_b = trial.suggest_int('C_b', C_a, 99)
    C_c = 99

    # --- THRESHOLDS ---
    d_threshold = trial.suggest_float('d_threshold', 0.1, 0.99)
    c_threshold = trial.suggest_float('c_threshold', 0.2, 0.9)

    params = {
    'D_a': D_a, 'D_b': D_b, 'D_c': D_c,
    'C_a': C_a, 'C_b': C_b, 'C_c': C_c,
    'd_threshold': d_threshold,
    'c_threshold': c_threshold,
    }

    try:
        fuzzy_player = build_fuzzy_player(params)
    except Exception as e:
        print(f"Failed to build player: {e}")
        return 0.0

    opponents = [player() for player in axelrod.stewart_plotkin_strategies]

    try:
        tournament = axelrod.Tournament(
            [fuzzy_player] + opponents,
            turns=200,
            repetitions = 5
        )
        results = tournament.play(progress_bar=False)
    except Exception as e:
        print(f"Tournament failed: {e}")
        return 0.0

    return np.mean(results.normalised_scores[0])

In [5]:
study = optuna.create_study(
    direction='maximize',
    study_name='fuzzy_optimization_full_start_c_d_fixed',
    storage='sqlite:///fuzzy_optuna_full_start_c_d_fixed.db',
    load_if_exists=True
)
study.enqueue_trial({
        'D_b': 25, 'D_c': 50,
        'C_a': 35, 'C_b': 75,
        'd_threshold': 0.4,
        'c_threshold': 0.6,
})
study.optimize(objective, n_trials=300, show_progress_bar=True)

print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Best score:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

importance = optuna.importance.get_param_importances(study)
print("\n=== PARAMETER IMPORTANCE ===")
for param, imp in importance.items():
    print(f"  {param}: {imp:.4f}")

[I 2026-03-05 11:38:55,451] Using an existing study with name 'fuzzy_optimization_full_start_c_d_fixed' instead of creating a new one.
Best trial: 1. Best value: 2.71193:   0%|          | 1/300 [00:15<1:17:47, 15.61s/it]

[I 2026-03-05 11:39:11,090] Trial 1 finished with value: 2.711928571428571 and parameters: {'D_b': 25, 'D_c': 50, 'C_a': 35, 'C_b': 75, 'd_threshold': 0.4, 'c_threshold': 0.6}. Best is trial 1 with value: 2.711928571428571.


Best trial: 1. Best value: 2.71193:   1%|          | 2/300 [00:28<1:10:05, 14.11s/it]

[I 2026-03-05 11:39:24,165] Trial 2 finished with value: 2.6065714285714288 and parameters: {'D_b': 3, 'D_c': 34, 'C_a': 51, 'C_b': 93, 'd_threshold': 0.9662103665042864, 'c_threshold': 0.4796795000118644}. Best is trial 1 with value: 2.711928571428571.


Best trial: 3. Best value: 2.71679:   1%|          | 3/300 [00:43<1:11:29, 14.44s/it]

...

[I 2026-03-05 13:36:44,640] Trial 298 finished with value: 2.6997142857142857 and parameters: {'D_b': 32, 'D_c': 33, 'C_a': 54, 'C_b': 55, 'd_threshold': 0.1506446630654853, 'c_threshold': 0.27965397864224245}. Best is trial 239 with value: 2.784357142857143.


Best trial: 239. Best value: 2.78436: 100%|█████████▉| 299/300 [1:58:05<00:18, 18.45s/it]

[I 2026-03-05 13:37:00,953] Trial 299 finished with value: 2.7470000000000003 and parameters: {'D_b': 27, 'D_c': 30, 'C_a': 53, 'C_b': 98, 'd_threshold': 0.1374949862163464, 'c_threshold': 0.5187509595698565}. Best is trial 239 with value: 2.784357142857143.


Best trial: 239. Best value: 2.78436: 100%|██████████| 300/300 [1:58:22<00:00, 23.67s/it]


[I 2026-03-05 13:37:17,847] Trial 300 finished with value: 2.6759999999999997 and parameters: {'D_b': 37, 'D_c': 38, 'C_a': 46, 'C_b': 92, 'd_threshold': 0.11869904875662028, 'c_threshold': 0.22776978644653637}. Best is trial 239 with value: 2.784357142857143.

=== OPTIMIZATION COMPLETE ===
Best score:  2.7844
Best params: {'D_b': 8, 'D_c': 9, 'C_a': 59, 'C_b': 60, 'd_threshold': 0.354659850655831, 'c_threshold': 0.2849997673688588}

=== PARAMETER IMPORTANCE ===
  D_b: 0.6587
  d_threshold: 0.2863
  c_threshold: 0.0358
  C_a: 0.0192
